# 🎯 X-Report 2-Stage 시스템 (완전판)

## 아키텍처
```
Evidence Pack (그대로 유지) ← 완벽함!
    ↓
Stage 1 (DeepSeek-R1): 데이터 분석 + 추론
    ↓
Stage 2 (DeepSeek-R1): 솔루션 3-5개 작성
    ↓
X-Report 완성
```

## 특징
- ✅ 완전 무료 (API 키 불필요)
- ✅ Evidence Pack 그대로 활용
- ✅ 2-Stage로 명확한 분리
- ✅ Stage2 안정적 (EXAONE 제거)
- ✅ 솔루션 3-5개 보장

## 1. 환경 설정

In [1]:
# GPU 확인
!nvidia-smi

Tue Feb  3 08:19:21 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          Off |   00000000:00:05.0 Off |                    0 |
| N/A   34C    P0             52W /  400W |       0MiB /  81920MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [2]:
!pip install -q transformers accelerate torch

In [3]:
import json
import os
import time
import re
from typing import Dict, List, Optional
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

print("✅ 로드 완료")
print(f"GPU: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"이름: {torch.cuda.get_device_name(0)}")

✅ 로드 완료
GPU: True
이름: NVIDIA A100-SXM4-80GB


## 2. 데이터 업로드

왼쪽 📁:
- `상권X매출X리뷰.json`
- `인구_DB.json`

In [4]:
MARKET_PATH = "/content/상권X매출X리뷰.json"
POP_PATH = "/content/인구_DB.json"

print(f"상권: {'✅' if os.path.exists(MARKET_PATH) else '❌'}")
print(f"인구: {'✅' if os.path.exists(POP_PATH) else '❌'}")

상권: ✅
인구: ✅


## 3. 데이터 로더 (기존 유지)

In [5]:
class DataLoader:
    AREA_MAP = {"망원1동": "11440690", "망원2동": "11440700"}

    def __init__(self, market_path, pop_path):
        print("📂 로딩...")
        with open(market_path, 'r', encoding='utf-8') as f:
            self.market = json.load(f)
        with open(pop_path, 'r', encoding='utf-8') as f:
            self.pop = json.load(f)
        self._build_index()
        print(f"✅ 매장: {len(self.stores)}개")

    def _build_index(self):
        self.stores = {}
        self.pops = {}
        for rec in self.market:
            for report in rec.get('analysis_reports', []):
                name = report.get('store_name', '')
                if name:
                    if name not in self.stores:
                        self.stores[name] = []
                    self.stores[name].append({
                        **report,
                        'market_context': rec
                    })
        for p in self.pop:
            self.pops[p.get('id', '')] = p

    def get_store(self, name):
        norm = name.replace(' ', '')
        if name in self.stores:
            return self.stores[name]
        if norm in self.stores:
            return self.stores[norm]
        for k in self.stores:
            if norm in k.replace(' ', ''):
                return self.stores[k]
        return None

    def get_pop(self, area, q="20253"):
        code = self.AREA_MAP.get(area, area)
        return self.pops.get(f"{code}_{q}_population")

    def list_stores(self, n=20):
        return list(self.stores.keys())[:n]

print("✅ DataLoader 완료")

✅ DataLoader 완료


## 4. Evidence Pack 생성기 (기존 유지)

In [6]:
class EvidenceBuilder:
    """기존 Evidence Pack 생성기 (그대로 유지)"""

    @staticmethod
    def build(store_data, pop_data):
        """
        기존 코드 그대로 사용
        출력 예시:

        ### [① 고객 리뷰 및 피드백] (비중: 50%)
        전체 감정: 긍정적 (점수 0.75)
        세부 평가:
          • taste: 매우 좋음 (0.85)
        ...
        """
        if not store_data:
            return "ERROR: No data"

        first = store_data[0]
        store_name = first.get('store_name', 'Unknown')
        market_ctx = first.get('market_context', {})
        metadata = market_ctx.get('metadata', {})

        lines = []
        lines.append("=" * 60)
        lines.append(f"EVIDENCE PACK: {store_name} (지역: {metadata.get('area', 'N/A')})")
        lines.append("=" * 60)
        lines.append("")

        # ① 리뷰
        lines.append("### [① 고객 리뷰 및 피드백] (비중: 50%)")
        lines.append("-" * 40)

        review = first.get('review_metrics', {})
        if review:
            overall = review.get('overall_sentiment', {})
            lines.append(f"전체 감정: {overall.get('label', 'N/A')} (점수 {overall.get('score', 0):.2f})")
            lines.append(f"→ {overall.get('comparison', '')}")

            lines.append("세부 평가:")
            features = review.get('feature_scores', {})
            for feat, data in features.items():
                score = data.get('score', 0)
                avg = data.get('avg_score', 0)
                diff = score - avg
                lines.append(f"  • {feat}: {data.get('label', 'N/A')} ({score:.2f}, 평균 대비 {diff:+.2f})")

            keywords = first.get('top_keywords', [])
            if keywords:
                lines.append(f"고객 언급 키워드: {', '.join(keywords)}")

            critical = first.get('critical_feedback', [])
            if critical:
                lines.append("⚠️ 개선 필요 사항:")
                for c in critical:
                    lines.append(f"  - {c}")

            rag = first.get('rag_context', '')
            if rag:
                lines.append("")
                lines.append("리뷰 종합 평가:")
                lines.append(rag)

        lines.append("")

        # ② 상권
        lines.append("### [② 상권 현황] (비중: 20%)")
        lines.append("-" * 40)
        market_analysis = market_ctx.get('market_analysis', {})
        lines.append(f"시장 단계  : {market_analysis.get('phase', 'N/A')}")
        lines.append(f"추이 리뷰  : {market_analysis.get('trend_review', 'N/A')}")
        lines.append(f"경쟁 강도  : {market_analysis.get('competition_intensity', 'N/A')}")
        lines.append("")

        # ③ 매출
        lines.append("### [③ 매출 분석] (비중: 20%)")
        lines.append("-" * 40)
        revenue = market_ctx.get('revenue_analysis', 'N/A')
        lines.append(revenue)
        lines.append("")

        # ④ 인구
        if pop_data:
            lines.append("### [④ 인구 특성] (비중: 10%)")
            lines.append("-" * 40)
            lines.append(f"지역: {metadata.get('area', 'N/A')}  |  분기: {metadata.get('quarter', 'N/A')}")
            pop_analysis = pop_data.get('population_analysis', {})
            lines.append(f"전체 현황  : {pop_analysis.get('overall_summary', 'N/A')}")
            lines.append(f"성별 구성  : {pop_analysis.get('gender_structure', 'N/A')}")
            lines.append(f"연령 구성  : {pop_analysis.get('age_structure', 'N/A')}")
            lines.append(f"시간대 패턴 : {pop_analysis.get('temporal_pattern', 'N/A')}")
            lines.append(f"직장 인구  : {pop_analysis.get('working_population_summary', 'N/A')}")

        lines.append("")
        lines.append("=" * 60)

        return "\n".join(lines)

print("✅ EvidenceBuilder 완료")

✅ EvidenceBuilder 완료


## 5. Stage 1: 분석 및 추론 (DeepSeek-R1)

In [7]:
class Stage1Analyzer:
    """DeepSeek-R1 기반 분석 및 추론"""

    SYSTEM = """당신은 데이터 분석 전문가입니다.

역할: Evidence Pack을 분석하여 **핵심 인사이트와 문제점**을 추출

분석 우선순위:
1. 고객 리뷰 (50%) - 개선 필요 사항, 낮은 점수
2. 상권 현황 (20%) - 경쟁 강도, 시장 단계
3. 매출 분석 (20%) - 객단가, 시간대 패턴
4. 인구 특성 (10%) - 타겟 기회
"""

    USER = """[Evidence Pack]
{evidence}

위 데이터를 분석하세요.

**먼저 <think> 태그로 분석:**
<think>
1. 가장 큰 문제는? (개선 필요 사항 확인)
2. 점수 약점은? (평균 대비 낮은 것)
3. 인구 데이터 기회는? (연령/시간대)
4. 우선순위는?
</think>

**그 다음 JSON 출력:**
{{
  "diagnosis": {{
    "main_problem": "가장 큰 문제 (1줄)",
    "strengths": ["강점1", "강점2"],
    "weaknesses": ["약점1", "약점2"]
  }},
  "insights": [
    {{
      "category": "리뷰|상권|매출|인구",
      "finding": "발견한 것",
      "evidence": "근거"
    }}
  ]
}}

insights는 3-5개만
"""

    def __init__(self, model_name="deepseek-ai/DeepSeek-R1-Distill-Qwen-14B"):
        print(f"🤖 Stage1 로딩: {model_name}")
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForCausalLM.from_pretrained(
            model_name,
            device_map="auto",
            torch_dtype=torch.float16,
            low_cpu_mem_usage=True
        )
        print("✅ Stage1 완료")

    def analyze(self, evidence):
        prompt = self.USER.format(evidence=evidence)
        messages = [
            {"role": "system", "content": self.SYSTEM},
            {"role": "user", "content": prompt}
        ]

        input_text = self.tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )

        inputs = self.tokenizer(
            input_text,
            return_tensors="pt",
            truncation=True,
            max_length=4096
        ).to(self.model.device)

        print("🧠 분석 중...")

        with torch.no_grad():
            outputs = self.model.generate(
                **inputs,
                max_new_tokens=2048,
                do_sample=False,
                pad_token_id=self.tokenizer.eos_token_id
            )

        generated_ids = outputs[0][inputs['input_ids'].shape[1]:]
        result = self.tokenizer.decode(generated_ids, skip_special_tokens=True)

        thinking = self._extract_thinking(result)
        parsed = self._parse_json(result)

        return {
            "thinking": thinking,
            "analysis": parsed
        }

    def _extract_thinking(self, text):
        match = re.search(r'<think>(.*?)</think>', text, re.DOTALL)
        return match.group(1).strip() if match else ""

    def _parse_json(self, text):
        try:
            text = re.sub(r'<think>.*?</think>', '', text, flags=re.DOTALL)
            text = re.sub(r'```json\s*', '', text)
            text = re.sub(r'```\s*', '', text)
            json_match = re.search(r'\{.*\}', text, re.DOTALL)
            if json_match:
                return json.loads(json_match.group(0))
            return json.loads(text)
        except Exception as e:
            print(f"⚠️ JSON 파싱 실패: {e}")
            return {
                "diagnosis": {
                    "main_problem": "분석 실패",
                    "strengths": [],
                    "weaknesses": []
                },
                "insights": []
            }

print("✅ Stage1Analyzer 완료")

✅ Stage1Analyzer 완료


## 6. Stage 2: 솔루션 작성 (DeepSeek-R1)

In [8]:
class Stage2Writer:
    """DeepSeek-R1 기반 솔루션 작성"""

    SYSTEM = """당신은 비즈니스 컨설턴트입니다.

역할: Stage1 분석을 바탕으로 **즉시 실행 가능한 솔루션 3-5개** 작성

솔루션 기준:
- 구체적: 누가, 언제, 어떻게
- 정량적: 숫자로 효과 명시
- 실행 가능: 이번 주 시작
- 우선순위: 중요한 것부터
"""

    USER = """[Stage1 분석]
{analysis}

위 분석을 바탕으로 솔루션을 작성하세요.

출력:

# 🎯 {store_name} 개선 리포트

## 📊 현황 진단
**핵심 문제**: (main_problem)

**강점**:
- (strengths)

**약점**:
- (weaknesses)

---

## 💡 실행 솔루션

### 🔥 솔루션 1: (제목)

**왜 중요?**
(insights에서 근거)

**무엇을?**
(한 줄)

**어떻게?**
1. 이번 주: ...
2. 다음 주: ...
3. 2주 후: ...

**예상 효과**
(정량적: "재방문율 15% 증가")

**비용**
(예상)

---

### ⭐ 솔루션 2:
(동일 형식)

### 🎯 솔루션 3:
(동일 형식)

---

## 📞 다음 단계
1. 이번 주: 솔루션 1 착수
2. 2주 후: 효과 측정
3. 효과 있으면: 솔루션 2 시작

**중요**: 솔루션 3-5개, 우선순위 명확
"""

    def __init__(self, model_name="deepseek-ai/DeepSeek-R1-Distill-Qwen-14B"):
        print(f"📝 Stage2 로딩: {model_name}")
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForCausalLM.from_pretrained(
            model_name,
            device_map="auto",
            torch_dtype=torch.float16,
            low_cpu_mem_usage=True
        )
        print("✅ Stage2 완료")

    def write(self, analysis, store_name):
        analysis_str = json.dumps(analysis, ensure_ascii=False, indent=2)
        prompt = self.USER.format(
            analysis=analysis_str,
            store_name=store_name
        )

        messages = [
            {"role": "system", "content": self.SYSTEM},
            {"role": "user", "content": prompt}
        ]

        input_text = self.tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )

        inputs = self.tokenizer(
            input_text,
            return_tensors="pt",
            truncation=True,
            max_length=4096
        ).to(self.model.device)

        print("✍️  솔루션 작성 중...")

        with torch.no_grad():
            outputs = self.model.generate(
                **inputs,
                max_new_tokens=3072,
                do_sample=False,
                pad_token_id=self.tokenizer.eos_token_id
            )

        generated_ids = outputs[0][inputs['input_ids'].shape[1]:]
        result = self.tokenizer.decode(generated_ids, skip_special_tokens=True)

        # # 로 시작하는 부분부터 추출
        match = re.search(r'(# .*)', result, re.DOTALL)
        if match:
            return match.group(1).strip()
        return result.strip()

print("✅ Stage2Writer 완료")

✅ Stage2Writer 완료


## 7. 파이프라인

In [9]:
class XReportPipeline:
    """2-Stage 파이프라인"""

    def __init__(self, market_path, pop_path):
        self.loader = DataLoader(market_path, pop_path)

        # 모델 로딩 (한 번만)
        print("\n🤖 모델 로딩 중 (5-10분 소요)...")
        self.stage1 = Stage1Analyzer()
        self.stage2 = Stage2Writer()
        print("\n✅ 파이프라인 준비 완료\n")

    def generate(self, name, debug=False):
        print("=" * 70)
        print(f"🏪 매장: {name}")
        print("=" * 70)

        # 데이터 조회
        store = self.loader.get_store(name)
        if not store:
            avail = self.loader.list_stores(10)
            raise ValueError(f"'{name}' 없음\n가능: {avail}")

        first = store[0]
        ctx = first.get('market_context', {})
        area = ctx.get('metadata', {}).get('area', '')
        pop = self.loader.get_pop(area)

        # Evidence Pack 생성
        print("\n📋 Evidence Pack 생성 중...")
        evidence = EvidenceBuilder.build(store, pop)

        if debug:
            print("\n[Evidence Pack 일부]")
            print(evidence[:500] + "...\n")

        # Stage 1
        print("🔍 Stage 1: 분석 및 추론...")
        t0 = time.time()
        stage1_result = self.stage1.analyze(evidence)
        t1 = time.time()
        print(f"  ✅ 완료 ({t1-t0:.1f}초)")

        if debug and stage1_result['thinking']:
            print("\n[Stage1 추론 과정]")
            print(stage1_result['thinking'][:300] + "...\n")

        # Stage 2
        print("✍️  Stage 2: 솔루션 작성...")
        t2 = time.time()
        report = self.stage2.write(
            stage1_result['analysis'],
            first.get('store_name', 'Unknown')
        )
        t3 = time.time()
        print(f"  ✅ 완료 ({t3-t2:.1f}초)")

        print(f"\n⏱️  총: {t3-t0:.1f}초")
        print("=" * 70)

        print("\n📄 X-REPORT:")
        print("=" * 70)
        print(report)
        print("=" * 70)

        return {
            "store": first.get('store_name'),
            "evidence": evidence,
            "stage1_thinking": stage1_result['thinking'],
            "stage1_analysis": stage1_result['analysis'],
            "report": report,
            "time": t3 - t0
        }

print("✅ XReportPipeline 완료")

✅ XReportPipeline 완료


## 8. 실행

In [10]:
# 초기화 (모델 로딩 5-10분)
pipeline = XReportPipeline(MARKET_PATH, POP_PATH)

📂 로딩...
✅ 매장: 721개

🤖 모델 로딩 중 (5-10분 소요)...
🤖 Stage1 로딩: deepseek-ai/DeepSeek-R1-Distill-Qwen-14B


config.json:   0%|          | 0.00/664 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


...

Loading weights:   0%|          | 0/579 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/181 [00:00<?, ?B/s]

✅ Stage1 완료
📝 Stage2 로딩: deepseek-ai/DeepSeek-R1-Distill-Qwen-14B


Loading weights:   0%|          | 0/579 [00:00<?, ?it/s]

✅ Stage2 완료

✅ 파이프라인 준비 완료



In [11]:
# 매장 목록
stores = pipeline.loader.list_stores(20)
print("📋 매장 (20개):")
for i, s in enumerate(stores, 1):
    print(f"  {i}. {s}")

📋 매장 (20개):
  1. 하노이엔
  2. 싱싱샐러드 망원점
  3. 돈까스참잘하는집
  4. 광
  5. 하치고
  6. 픽셀
  7. 소소베이크하우스
  8. 술빵술찐빵
  9. 스위즈
  10. 다시점
  11. 소이양꼬치
  12. 설경
  13. BBQ 망원점
  14. 광계토 숯불바베큐
  15. 동근이숯불두마리치킨 서울망원점
  16. 교촌치킨 망원2동점
  17. 페리카나 망원동점
  18. 썬더치킨 망원점
  19. 정드린치킨 망원점
  20. 달리는커피 서울망원점


In [12]:
# 생성
NAME = "장터국밥"  # 원하는 매장명

result = pipeline.generate(NAME, debug=True)

The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


🏪 매장: 장터국밥

📋 Evidence Pack 생성 중...

[Evidence Pack 일부]
EVIDENCE PACK: 장터국밥 (지역: 망원2동)

### [① 고객 리뷰 및 피드백] (비중: 50%)
----------------------------------------
전체 감정: 긍정적 (점수 0.75)
→ 평균 대비 긍정적인 반응을 보임
세부 평가:
  • taste: 매우 좋음 (0.85, 평균 대비 +0.15)
  • price_value: 보통 (0.60, 평균 대비 +0.02)
  • cleanliness: 매우 좋음 (0.80, 평균 대비 +0.08)
  • service: 좋음 (0.70, 평균 대비 +0.02)
고객 언급 키워드: 순대국, 진국, 아날로그 감성, 망원시장, 포장 판매, 친절한 사장님
⚠️ 개선 필요 사항:
  - 음식의 ...

🔍 Stage 1: 분석 및 추론...
🧠 분석 중...
  ✅ 완료 (59.7초)
✍️  Stage 2: 솔루션 작성...
✍️  솔루션 작성 중...
  ✅ 완료 (85.9초)

⏱️  총: 145.6초

📄 X-REPORT:
# 🎯 장터국밥 개선 리포트

## 📊 현황 진단
**핵심 문제**: 음식의 국물이 맹물 맛이 날 수 있어 개인 취향에 따른 양념장 사용이 필수적

**강점**:
- 고객 리뷰에서 맛과 청결도, 서비스가 긍정적으로 평가됨
- 경쟁 강도가 중간 수준으로 안정적인 시장 입지

**약점**:
- 가격 대비 가치가 평균 대비 약간 낮음
- 일부 고객이 아날로그 분위기가 불편할 수 있음

---

## 💡 실행 솔루션

### 🔥 솔루션 1: 양념장 제공 개선 및 맞춤형 서비스 도입

**왜 중요?**  
고객들은 국물 맛이 시원하지만 맹물 맛이 날 수 있어 양념장 사용이 필수적이라고 피드백했습니다. 또한, 가격 대비 가치가 평균 대비 약간 낮은 점을 개선할 수 있습니다.

**무엇을?**  
고객 맞춤형 양념장 제공 및 기본 양념장 퀄리티 향상.

**어떻게?**  
1

In [13]:
# 저장
if 'result' in locals():
    with open(f"/content/xreport_{NAME}.json", 'w', encoding='utf-8') as f:
        json.dump(result, f, ensure_ascii=False, indent=2)

    with open(f"/content/xreport_{NAME}.md", 'w', encoding='utf-8') as f:
        f.write(result['report'])

    print(f"💾 저장 완료")
    print(f"📥 왼쪽에서 다운로드")

💾 저장 완료
📥 왼쪽에서 다운로드


## 9. 배치 실행

In [14]:
# 여러 매장
NAMES = ["장터국밥", "하노이엔", "싱싱샐러드 망원점"]

results = []
for name in NAMES:
    try:
        result = pipeline.generate(name, debug=False)
        results.append(result)

        with open(f"/content/xreport_{name}.md", 'w', encoding='utf-8') as f:
            f.write(result['report'])

        print(f"\n✅ {name} 완료")
    except Exception as e:
        print(f"\n❌ {name} 실패: {e}")

print(f"\n✅ 완료: {len(results)}개")
if results:
    print(f"평균: {sum(r['time'] for r in results) / len(results):.1f}초")

🏪 매장: 장터국밥

📋 Evidence Pack 생성 중...
🔍 Stage 1: 분석 및 추론...
🧠 분석 중...
  ✅ 완료 (57.8초)
✍️  Stage 2: 솔루션 작성...
✍️  솔루션 작성 중...
  ✅ 완료 (86.1초)

⏱️  총: 143.9초

📄 X-REPORT:
# 🎯 장터국밥 개선 리포트

## 📊 현황 진단
**핵심 문제**: 음식의 국물이 맹물 맛이 날 수 있어 개인 취향에 따른 양념장 사용이 필수적

**강점**:
- 고객 리뷰에서 맛과 청결도, 서비스가 긍정적으로 평가됨
- 경쟁 강도가 중간 수준으로 안정적인 시장 입지

**약점**:
- 가격 대비 가치가 평균 대비 약간 낮음
- 일부 고객이 아날로그 분위기가 불편할 수 있음

---

## 💡 실행 솔루션

### 🔥 솔루션 1: 양념장 제공 개선 및 맞춤형 서비스 도입

**왜 중요?**  
고객들은 국물 맛이 시원하지만 맹물 맛이 날 수 있어 양념장 사용이 필수적이라고 피드백했습니다. 또한, 가격 대비 가치가 평균 대비 약간 낮은 점을 개선할 수 있습니다.

**무엇을?**  
고객 맞춤형 양념장 제공 및 기본 양념장 퀄리티 향상.

**어떻게?**  
1. 이번 주:  
   - 기본 양념장 맛 개선 ( пря한, 달콤한, 짠口味 중에서 선택 )  
   - 고객 맞춤형 양념장 제공 (손님이 원하는 맛을 선택할 수 있는 QR 코드를 통해 주문 )  
2. 다음 주:  
   - 양념장 제공을 위한 별도의 소품 (컵, 스푸oons) 준비  
   - 직원 교육 (양념장 제공 방법 및 고객 서비스)  
3. 2주 후:  
   - 고객 피드백을 바탕으로 양념장 맛 최종 조정  

**예상 효과**  
- 재방문율 15% 증가  
- 긍정적 리뷰 20% 증가  

**비용**  
- 양념장 개발 및 소품 구입: 50만 원  
- 교육 및 인력 배치: 10만 원  

---

### ⭐ 솔루션 2: 디지털 결제 및 예약 시스템 도입

**왜 중요?**  
유동 인구가 

## 10. 사용 가이드

### GPU
- A100 필수 (28GB)
- 설정: 런타임 → A100

### 소요 시간
- 모델 로딩: 5-10분 (처음 1회)
- Stage1: 1-2분
- Stage2: 1-2분
- 매장당: 2-4분

### 특징
- ✅ 완전 무료
- ✅ Evidence Pack 그대로
- ✅ 2-Stage 명확
- ✅ 솔루션 3-5개
- ✅ Stage2 안정적 (EXAONE 제거)

### 아키텍처
```
Evidence Pack
    ↓
Stage1 (DeepSeek-R1):
  - 데이터 분석
  - <think> 추론
  - JSON 구조화
    ↓
Stage2 (DeepSeek-R1):
  - JSON 기반
  - 솔루션 3-5개
  - 마크다운 출력
```